# Datamine QNLM Process Example

This notebook demonstrates how to configure and run the **`qnlm`** process wrapper in `dmstudio`.

## Process Description

## QNLM

Q - mode non linear mapping groups together samples using the euclidian distance to calculate a dissimilarity matrix.

As such, it groups samples on the basis of distance (element dissimilarity) between all pairs of samples in multi-dimensional space. These distances are recalculated into two dimensional space for graphical output.

[![](../Images/QNLM.jpg)](<javascript:void\(0\);>)

Note the difference with R mode non linear mapping analysis which clusters elements or parameters.

A two dimensional view of the sample clusters, or scores **NLM-X** versus **NLM-Y** , is calculated to represent them as they would appear in multi- dimensional space on the basis of their dissimilarity calculated from each other for every pair of samples (see Figure 4). All input data can be standardized (default) prior to calculation of the matrix. Similarly, the output scores can also be normalized prior to plotting (default).

Q - mode non linear mapping, when compared with Q - mode factor analysis, is more sophisticated and tends not to distort or sub-divide the clustering of samples. Experience has shown that non - linear mapping will give more separable clusters than hierarchical techniques such as Q - mode cluster analysis.

### File Handling

The input file &(**IN**) must have a separate identifier field (* **SAMPID**). The output file &(**SCORES**) contains three parameters, **FIELD** the sample identifier, **NLM-X** and **NLM-Y** the output scores for plotting the sample distances in multi-dimensional space. Results can be displayed using **QUIG** , **PLOTAN** or **PLOTDA**.

### Iteration Procedure

In order to present a two dimensional view of multi-dimensional space with minimum distortion of the sample clusters, the calculated mapping error has to be minimized by an iterative method (steepest descent). This is controlled by the @CONVLIM parameter, that is the minimum difference allowed in the mapping error between iterations, @MAXIT, the maximum number of iterations permitted and the @MAGIC parameter which specifies the stepping function used for each iteration. If the stepping function @MAGIC is decreased, the number of iterations is increased with an obvious time penalty on the length of the calculation. The value used must be taken into account when there are a large number of samples or when using a PC. However the results can be more stable.

### Input Files:

* **in** (Undefined):
  Input file.
  Required=Yes

### Output Files:

* **scores** (Undefined):
  Optional output file for non linear mapping scores.
  Required=No

### Fields:

* **sampid** (Any : IN):
  Sample identifier field in input file.
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  First field to be used. No fields specified means all.
  Default=Undefined
  Required=No

### Parameters:

* **convlim**:
  Convergence limit (0.0001)
  Range=Undefined
  Values=Undefined
  Default=0.0001
  Required=No

* **magic**:
  Convergence [magic] factor (0.35)
  Range=Undefined
  Values=Undefined
  Default=0.35
  Required=No

* **maxit**:
  Maximum number of iterations (100)
  Range=Undefined
  Values=Undefined
  Default=100
  Required=No

* **standard**:
  >0 Input data to be standardised (0)
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **znorm**:
  >0 NLM scores in output file SCORES to be Z normalised (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:
  >0 Display two dimensional x - y coordinates to the screen (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('qnlm')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute qnlm
print("Running qnlm...")
dm_cmd.qnlm(
    in_i='t_assays',  # required
    scores_o=['t_qnlm_out'],  # required
    sampid_f='optional',  # required
    fields_f=['AU'],  # required
    # convlim_p=0.0001,  # optional
    # magic_p=0.35,  # optional
    # maxit_p=100,  # optional
    # standard_p=0,  # optional
    # znorm_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("qnlm execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_qnlm_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")